In [1]:
import re
from pathlib import Path

import pandas as pd

if 'df' not in globals():
    excel_path = Path('/Users/padrian/Documents/08_Tools/51_Inventar_Luzern/data/Bauten_ab_1970.xlsx')
    if not excel_path.exists():
        raise FileNotFoundError(f'Excel-Datei nicht gefunden: {excel_path}')
    df = pd.read_excel(excel_path)
    print(f'Geladene Zeilen: {len(df)}')

project_root = Path('/Users/padrian/Documents/08_Tools/51_Inventar_Luzern')
output_dir = project_root / 'output' / 'google_maps_screenshots'
output_dir.mkdir(parents=True, exist_ok=True)

def safe_slug(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return 'unknown'
    text = str(value)
    text = text.replace('&', 'and').replace('/', '_')
    text = re.sub(r'[^0-9A-Za-zÄÖÜäöüß_ -]', '', text)
    text = text.strip().replace(' ', '_')
    return text[:40] or 'unknown'

print('df vorhanden:', 'df' in globals())
print('output_dir:', output_dir)


Geladene Zeilen: 171
df vorhanden: True
output_dir: /Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/google_maps_screenshots


In [2]:
# pip install geopy playwright

from urllib.parse import quote_plus

try:
    from geopy.geocoders import Nominatim
    have_geopy = True
except Exception:
    have_geopy = False


def geocode_address(address):
    if not have_geopy:
        return None, None

    query = str(address).strip()
    if not query:
        return None, None

    try:
        locator = Nominatim(user_agent='lucerne_maps_screenshots')
        loc = locator.geocode(f"{query}, Schweiz", exactly_one=True, timeout=10)
        if loc is None:
            return None, None
        return loc.latitude, loc.longitude
    except Exception:
        return None, None


def build_full_address(row):
    street = str(row.get('STRNAMK1_HPT', '')).strip()
    nr = str(row.get('DEINR', '')).strip()
    gemeinde = str(row.get('Gemeinde', '')).strip()
    plz = str(row.get('PLZ', '')).strip()

    parts = []
    if street and nr:
        parts.append(f"{street} {nr}")
    elif street:
        parts.append(street)
    elif nr:
        parts.append(nr)

    if plz:
        parts.append(plz)
    if gemeinde:
        parts.append(gemeinde)

    address = ', '.join(parts)
    if address:
        return f"{address}, Schweiz"
    return None


def build_google_maps_perspective_url(row, camera='216a,35y,39.08t'):
    full_address = build_full_address(row)
    if not full_address:
        return None

    lat, lon = geocode_address(full_address)
    if lat is None or lon is None:
        return None

    place_name = full_address.replace(', Schweiz', '')
    return (
        f"https://www.google.com/maps/place/"
        f"{quote_plus(place_name)}/"
        f"@{lat},{lon},{camera}/"
        f"data=!3m1!1e3"
    )


preview = df[df['Adresse'].notna()].copy()
preview['full_address'] = preview.apply(build_full_address, axis=1)
preview['maps_url_perspective'] = preview.apply(build_google_maps_perspective_url, axis=1)
preview['screenshot_file'] = preview.apply(
    lambda r: output_dir / f"{safe_slug(r.get('Gemeinde', ''))[:20]}_{safe_slug(r.get('STRNAMK1_HPT', ''))[:30]}_{safe_slug(r.get('DEINR', ''))}.png"
    if pd.notna(r.get('DEINR')) else output_dir / 'unknown.png',
    axis=1,
)

preview[['Gemeinde', 'Adresse', 'full_address', 'maps_url_perspective', 'screenshot_file']].head(10)

# Beispiel nur ein Objekt testen
example_row = preview[preview['STRNAMK1_HPT'].fillna('').str.contains('Alpenquai', case=False)
                     & preview['DEINR'].astype(str).str.contains('12', na=False)].head(1)
example_url = example_row['maps_url_perspective'].iloc[0] if not example_row.empty else None
print('Beispiel-URL:', example_url)

try:
    from playwright.async_api import async_playwright
    have_playwright = True
except Exception:
    have_playwright = False

print('Playwright verfügbar:', have_playwright)
print('Geopy verfügbar:', have_geopy)

sample = preview[preview['maps_url_perspective'].notna()].head(5).copy()


async def accept_google_consent(page):
    candidates = [
        'Alle akzeptieren',
        'Accept all',
        'I agree',
        'Akzeptieren',
        'Zustimmen',
        'Ich stimme zu',
    ]
    for label in candidates:
        try:
            button = page.get_by_role('button', name=label)
            if await button.count():
                await button.first.click(timeout=2000)
                await page.wait_for_timeout(2000)
                return True
        except Exception:
            continue
    return False


async def capture_screenshots(rows):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(viewport={'width': 1400, 'height': 900}, locale='de-CH')
        context.set_default_timeout(60000)
        await context.add_cookies([
            {
                'name': 'CONSENT',
                'value': 'YES+1',
                'domain': '.google.com',
                'path': '/',
                'secure': True,
                'httpOnly': False,
                'sameSite': 'Lax',
            }
        ])
        page = await context.new_page()

        for _, row in rows.iterrows():
            url = row['maps_url_perspective']
            if pd.isna(url) or not url:
                continue

            await page.goto(url, wait_until='domcontentloaded', timeout=60000)
            await accept_google_consent(page)
            await page.wait_for_timeout(4000)
            await page.screenshot(path=str(row['screenshot_file']), full_page=True)

        await context.close()
        await browser.close()


if have_playwright and not sample.empty:
    await capture_screenshots(sample)
else:
    print('Keine Screenshot-Automation installiert oder keine gültigen Adressen für Screenshots.')

sample[['Gemeinde', 'Adresse', 'full_address', 'maps_url_perspective', 'screenshot_file']].head()


KeyError: 'STRNAMK1_HPT'